# `GenVarLoader`

In [ ]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import os
os.chdir(os.path.dirname(os.path.abspath('.')))
import pandas as pd
import polars as pl
import seqpro as sp
import genvarloader as gvl


import src.utils as utils
import src.genvarloader as GVL
import src.onekg as og
# import src.pyensembl as PYE

In [78]:
vcf_files = og.download_vcfs(skip_checks=True)

  return datetime.utcnow().replace(tzinfo=utc)



























































































































































































































































































































































































































































































































































































































































































































































































































































































































































































KeyboardInterrupt: 

In [93]:
og.list_remote_vcf(key='Human Genome Diversity Project')['local']

/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


0     /grid/koo/home/schilder/projects/data/1KG/vcf/...
1     /grid/koo/home/schilder/projects/data/1KG/vcf/...
2     /grid/koo/home/schilder/projects/data/1KG/vcf/...
3     /grid/koo/home/schilder/projects/data/1KG/vcf/...
4     /grid/koo/home/schilder/projects/data/1KG/vcf/...
5     /grid/koo/home/schilder/projects/data/1KG/vcf/...
6     /grid/koo/home/schilder/projects/data/1KG/vcf/...
7     /grid/koo/home/schilder/projects/data/1KG/vcf/...
8     /grid/koo/home/schilder/projects/data/1KG/vcf/...
9     /grid/koo/home/schilder/projects/data/1KG/vcf/...
10    /grid/koo/home/schilder/projects/data/1KG/vcf/...
11    /grid/koo/home/schilder/projects/data/1KG/vcf/...
12    /grid/koo/home/schilder/projects/data/1KG/vcf/...
13    /grid/koo/home/schilder/projects/data/1KG/vcf/...
14    /grid/koo/home/schilder/projects/data/1KG/vcf/...
15    /grid/koo/home/schilder/projects/data/1KG/vcf/...
16    /grid/koo/home/schilder/projects/data/1KG/vcf/...
17    /grid/koo/home/schilder/projects/data/1KG/

In [2]:
example_paths = GVL.prepare_example()

In [ ]:
bed = gvl.read_bedlike(example_paths['bed'])
bed.head()

In [ ]:
chrom = "chr22"
ds_path = os.path.join(GVL.DATA_DIR, f"{chrom}.gvl")
# bed_chr = bed.filter(pl.col("chrom")==chrom)

force = True
if not os.path.exists(ds_path) or force:
    print("Creating GVL database ==>",ds_path)
    gvl.write(
        path=ds_path,
        bed=example_paths['bed'],
        variants=example_paths['pgen'],
        # max_mem=16*2**30,
        overwrite=force
    )

In [ ]:
example_paths['reference']

In [41]:
ds = gvl.Dataset.open(ds_path, 
                      reference=example_paths['reference'])\
                      .with_seqs("annotated")

# n_train = round(ds.n_samples * 0.8)
# gene1_train_ds = ds.subset_to(samples=slice(0, n_train))
# dl = gene1_train_ds.to_dataloader(batch_size=16, num_workers=0, shuffle=True)

In [ ]:
ds = gvl.get_dummy_dataset().with_seqs("annotated")
ds

In [ ]:
haps = ds[0, :3] 
haps

In [ ]:
ds.haplotype_lengths()

## `cyvcf2`

In [ ]:
import numpy as np
from cyvcf2 import VCF

def vcf_to_genotype_matrix(vcf_file, region):
    """
    Converts a VCF file to a genotype matrix for a specific region.

    Args:
        vcf_file (str): Path to the VCF file.
        region (str): Region to extract (e.g., "chr1:1000-2000").

    Returns:
        tuple: A tuple containing:
            - genotype_matrix (numpy.ndarray): Genotype matrix (samples x variants).
            - variant_ids (list): List of variant IDs.
            - sample_ids (list): List of sample IDs.
    """
    vcf = VCF(vcf_file)
    sample_ids = vcf.samples
    genotype_matrix = []
    variant_ids = []

    for variant in vcf(region):
        variant_ids.append(variant.ID)
        genotypes = [
            sum(gt) if gt != [-1, -1] else np.nan for gt in variant.genotypes
        ]
        genotype_matrix.append(genotypes)

    vcf.close()
    return np.array(genotype_matrix).T, variant_ids, sample_ids

vcf_file_path = "/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
genotype_matrix, sample_names, variant_ids = vcf_to_genotype_matrix(vcf_file_path, "22:10664523-10684523")
print("Genotype Matrix:")
print(genotype_matrix)
print("\nSample Names:", sample_names)
print("\nVariant IDs:", variant_ids)

## `sgkit`

In [4]:
import sgkit as sg

In [5]:
ds = sg.simulate_genotype_call_dataset(n_variant=100, n_sample=50, missing_pct=.1)

In [ ]:
ds[['variant_allele', 'call_genotype']]


In [ ]:
ds.keys()

In [ ]:
df = ds.to_dataframe().reset_index()
df.head()

In [ ]:
# Create a new column by combining position and allele information
df['variant_id'] = df['variant_position'].astype(str) + df['variant_allele'].astype(str) + df['call_genotype'].astype(str)

# Create a ploidy-specific sample_id column
df['sample_id_ploidy'] = df['sample_id'].astype(str) + '_' + df['ploidy'].astype(str)

df.head()

Create a string representing the set of variants present in a given sample (one per ploid).

Then encode that string as an md5 hash for compression.

In [ ]:
import hashlib
df_agg = df.groupby('sample_id_ploidy')['variant_id'].apply(lambda x: ','.join(x)).reset_index()
df_agg['haplotype_hash'] = df_agg['variant_id'].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
df_agg.head()


Create a hashmap for each haplotype hash to a list of sample_ids.

In [ ]:
# Group by haplotype_hash and get a list of sample_ids for each hash
haplotype_samples = df_agg.groupby('haplotype_hash')['sample_id_ploidy'].apply(list).reset_index()
print(haplotype_samples.shape)
haplotype_samples

In [ ]:
sg.display_genotypes(ds, max_variants=10, max_samples=10)

In [ ]:
sg.convert_call_to_index(ds).call_genotype_index.values


## `haptools`

`haptools` provides some convenient data structures and functions for working with haplotypes. BUT it does not actually create haplotype file for you.  
Users must generate these themselves beforehand and figure out how to convert it into the haptools-specific `.hap` file format.

PLINK can generate haplotype blocks, but this is a data-driven approach (using linkage disequilibrium-clumping information from a popuatlion) that does not produce one block of a given window size (as models like SpliceAI expect).
https://www.cog-genomics.org/plink/1.9/ld#blocks

In [12]:
from haptools import data

In [13]:
vcf_file_path="/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
gt = data.GenotypesVCF(vcf_file_path) 

In [ ]:
print(gt.variants[:10])
print(gt.samples[:10])

In [ ]:
genotypes = gt.read(region="22:10664523-10764523")

In [ ]:
for i,line in enumerate(gt.__iter__(region="22:10664523-10764523")):
    print(line)
    if i > 5:
        break

In [ ]:
!haptools transform --region "22:10664523-10764523" {vcf_file_path} {vcf_file_path.replace('.vcf.gz', '.hap')}
